<a href="https://colab.research.google.com/github/KaeRuss/KaeRuss/blob/main/YouTube_Channel_Performance_Map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-api-python-client youtube-transcript-api langchain-community chromadb openai

In [ ]:
!pip install --upgrade youtube-transcript-api isodate
import youtube_transcript_api
from youtube_transcript_api import YouTubeTranscriptApi as YTA
import isodate

In [ ]:
!pip install isodate langchain langchain-text-splitters langchain-community

In [ ]:
!pip install isodate langchain langchain-text-splitters langchain-community

In [ ]:
from googleapiclient.discovery import build
from google.colab import sessions

# --- 1. SETUP YOUR CREDENTIALS ---

API_KEY = userdata.get('YT_TOKEN')
CHANNEL_ID = "UCn0AlJpkeq3w7TwnMTQ_zuQ" # Make sure this starts with UC

youtube = build('youtube', 'v3', developerKey=API_KEY)

# This part asks YouTube: "Where are this channel's uploads?"
channel_response = youtube.channels().list(
    id=CHANNEL_ID,
    part='contentDetails'
).execute()

# This is the "UU" ID you need!
uploads_id = channel_response['items'][0]['contentDetails']['relatedPlaylists']['uploads']
print(f"Your official Uploads Playlist ID is: {uploads_id}")

# Now, use that ID to list the videos
playlist_items = youtube.playlistItems().list(
    playlistId=uploads_id,
    part='snippet',
    maxResults=10
).execute()

for item in playlist_items['items']:
    print(f"Found Video: {item['snippet']['title']}")

In [ ]:
!pip install isodate
import isodate
import pandas as pd
import plotly.express as px
from googleapiclient.discovery import build
from google.colab import sessions

# --- 1. SETUP YOUR CREDENTIALS ---
API_KEY = userdata.get('YT_TOKEN')
CHANNEL_ID = "UCn0AlJpkeq3w7TwnMTQ_zuQ" # Starts with UC...

youtube = build('youtube', 'v3', developerKey=API_KEY)

# --- 2. FIND YOUR UPLOADS PLAYLIST ---
channel_response = youtube.channels().list(
    id=CHANNEL_ID,
    part='contentDetails'
).execute()
uploads_id = channel_response['items'][0]['contentDetails']['relatedPlaylists']['uploads']

# --- 3. FETCH ALL VIDEOS (THE FULL SCAN) ---
all_video_metrics = []
next_page_token = None

print("Scanning your entire channel for long-form videos... Please wait.")

while True:
    # Fetch a page of 50 videos
    playlist_res = youtube.playlistItems().list(
        playlistId=uploads_id,
        part='snippet,contentDetails',
        maxResults=50,
        pageToken=next_page_token
    ).execute()

    # Get IDs for the current batch
    v_ids = [item['contentDetails']['videoId'] for item in playlist_res['items']]

    # Get details (stats + duration) for the batch
    v_details = youtube.videos().list(
        part="snippet,statistics,contentDetails",
        id=",".join(v_ids)
    ).execute()

    for v_info in v_details['items']:
        v_title = v_info['snippet']['title']

        # Duration Filter (Skip Shorts)
        duration_iso = v_info['contentDetails']['duration']
        duration_seconds = isodate.parse_duration(duration_iso).total_seconds()

        if duration_seconds >= 60:
            v_stats = v_info['statistics']
            all_video_metrics.append({
                "Title": v_title,
                "Views": int(v_stats.get('viewCount', 0)),
                "Likes": int(v_stats.get('likeCount', 0)),
                "Comments": int(v_stats.get('commentCount', 0)),
                "Date": v_info['snippet']['publishedAt']
            })

    # Check for next page
    next_page_token = playlist_res.get('nextPageToken')
    if not next_page_token:
        break

# --- 4. CREATE TABLE AND GRAPH ---
df = pd.DataFrame(all_video_metrics)
print(f"\nSuccess! Found {len(df)} long-form videos.")

# Generate the 3D Plot
fig = px.scatter_3d(
    df,
    x='Comments',
    y='Likes',
    z='Views',
    color='Views',          # Color intensity changes with views
    hover_name='Title',     # Show title on mouse hover
    title='YouTube Full-Channel Performance Map',
    labels={'Views': 'Views', 'Likes': 'Likes', 'Comments': 'Comments'}
)

fig.update_traces(marker=dict(size=7)) # Dot size
fig.show()

Scanning your entire channel for long-form videos... Please wait.

Success! Found 30 long-form videos.
